# Task 1.1



In [ ]:
import csv
import json

with open("/content/appointments.csv", "r", newline="")as file :
  reader = csv.DictReader(file)

  rows = list(reader)

  print('header:', reader.fieldnames)
  print('Total Rows :', len(rows))

header: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Total Rows : 1020


In [ ]:
def parse_row(row):
    """
    Purpose: Converts numeric columns in one appointment row.

    Parameter:
        row (dict): One appointment record from the CSV.

    Returns:
        dict: The same row with converted numeric values.
    """
    row["age"] = int(row["age"])
    row["fee_inr"] = int(row["fee_inr"])
    row["prior_no_shows"] = int(row["prior_no_shows"])
    row["distance_km"] = float(row["distance_km"])

    return row

In [ ]:
accepted = []
rejected = []

for row in rows:

    try:
        converted = parse_row(row)
        accepted.append(converted)

    except ValueError:
        rejected.append(row)

print("Accepted:", len(accepted))
print("Rejected:", len(rejected))
print("Total:", len(accepted) + len(rejected))

Accepted: 988
Rejected: 32
Total: 1020


In [ ]:
with open("rejected_rows.json", "w") as file:

    json.dump(rejected, file, indent=2)

print("Rejected records saved successfully:", len(rejected))

Rejected records saved successfully: 32


# Task 1.2



In [ ]:
import pandas as pd

appointments = pd.read_csv("/content/appointments.csv")

print("Shape:", appointments.shape)


print("\nDataset Information:")
appointments.info()

print("\nMissing Values:")
print(appointments.isnull().sum())

print("\nExact Duplicates:")
print(appointments.duplicated().sum())

print("\nInitial Row Count:", len(appointments))

Shape: (1020, 14)

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB

Missing Values:
appointment_id      0
patient_id   

In [ ]:
appointments = appointments.drop_duplicates()

print("Rows after dropping duplicates:", len(appointments))


appointments["booking_channel"] = appointments["booking_channel"].str.strip().str.title()

print("Rows after cleaning booking_channel:", len(appointments))
print("Unique booking channels:", appointments["booking_channel"].unique())

Rows after dropping duplicates: 1000
Rows after cleaning booking_channel: 1000
Unique booking channels: ['Phone' 'Walk-In' 'App']


In [ ]:
appointments["appt_date"] = pd.to_datetime(appointments["appt_date"], errors="coerce")

appointments["booked_on"] = pd.to_datetime(appointments["booked_on"])

print("Rows after date conversion:", len(appointments))

Rows after date conversion: 1000


In [ ]:
appointments = appointments.dropna(subset=["appt_date", "status"])

print("Rows after dropping missing dates/status:", len(appointments))

Rows after dropping missing dates/status: 990


# Task 1.3


In [ ]:
import numpy as np

invalid_age = (appointments["age"] < 1) | (appointments["age"] > 100)

print("Invalid ages:", invalid_age.sum())

appointments.loc[invalid_age, "age"] = np.nan

print("Missing ages:", appointments["age"].isnull().sum())

Invalid ages: 5
Missing ages: 17


In [ ]:
age_median = appointments["age"].median()
distance_median = appointments["distance_km"].median()

print("Median Age:", age_median)
print("Median Distance:", distance_median)


appointments["age"] = appointments["age"].fillna(age_median).round().astype(int)


appointments["distance_km"] = appointments["distance_km"].fillna(distance_median)


print("Missing Age:", appointments["age"].isnull().sum())
print("Missing Distance:", appointments["distance_km"].isnull().sum())

Median Age: 48.0
Median Distance: 8.25
Missing Age: 0
Missing Distance: 0


In [ ]:
distance_q1 = appointments["distance_km"].quantile(0.25)
distance_q3 = appointments["distance_km"].quantile(0.75)
distance_iqr = distance_q3 - distance_q1


distance_lower = distance_q1 - 1.5 * distance_iqr
distance_upper = distance_q3 + 1.5 * distance_iqr


fee_q1 = appointments["fee_inr"].quantile(0.25)
fee_q3 = appointments["fee_inr"].quantile(0.75)
fee_iqr = fee_q3 - fee_q1


fee_lower = fee_q1 - 1.5 * fee_iqr
fee_upper = fee_q3 + 1.5 * fee_iqr


print("Distance:", distance_q1, distance_q3, distance_iqr)
print("Distance limits:", distance_lower, distance_upper)

print("Fee:", fee_q1, fee_q3, fee_iqr)
print("Fee limits:", fee_lower, fee_upper)

Distance: 4.625 13.0 8.375
Distance limits: -7.9375 25.5625
Fee: 500.0 787.5 287.5
Fee limits: 68.75 1218.75


In [ ]:
np.clip?

In [ ]:
distance_capped = (
    (appointments["distance_km"] < distance_lower) |
    (appointments["distance_km"] > distance_upper)
).sum()


fee_capped = (
    (appointments["fee_inr"] < fee_lower) |
    (appointments["fee_inr"] > fee_upper)
).sum()


appointments["distance_km"] = np.clip(
    appointments["distance_km"], distance_lower, distance_upper
)

appointments["fee_inr"] = np.clip(
    appointments["fee_inr"], fee_lower, fee_upper
)


print("Distance values capped:", distance_capped)
print("Fee values capped:", fee_capped)

Distance values capped: 26
Fee values capped: 3


In [ ]:
appointments["distance_km"] = appointments["distance_km"].round(1)


appointments["fee_inr"] = appointments["fee_inr"].round().astype(int)


print("Largest fee:", appointments["fee_inr"].max())

print("Final rows:", len(appointments))

print(appointments['distance_km'].dtypes, appointments['fee_inr'].dtypes)

Largest fee: 1219
Final rows: 990
float64 int64


# Task 1.4


In [ ]:
appointments["lead_days"] = (
    appointments["appt_date"] - appointments["booked_on"]
).dt.days


print(appointments["lead_days"].head())


print("Minimum lead days:", appointments["lead_days"].min())
print("Maximum lead days:", appointments["lead_days"].max())

0     1
1    10
2     6
4     4
5    40
Name: lead_days, dtype: int64
Minimum lead days: 1
Maximum lead days: 45


In [ ]:
appointments["no_show"] = (appointments["status"] == "No-Show").astype(int)


print(appointments[["status", "no_show"]].head())

print(appointments["no_show"].value_counts())



     status  no_show
0   No-Show        1
1  Attended        0
2  Attended        0
4  Attended        0
5   No-Show        1
no_show
0    772
1    218
Name: count, dtype: int64


In [ ]:
appointments["no_show"] = (appointments["status"] == "No-Show").astype(int)


print(appointments[["status", "no_show"]].head())

print(appointments["no_show"].value_counts())

     status  no_show
0   No-Show        1
1  Attended        0
2  Attended        0
4  Attended        0
5   No-Show        1
no_show
0    772
1    218
Name: count, dtype: int64


In [ ]:
X = appointments[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()


print(X)

print("Array shape:", X.shape)

[[ 60.   18.  850.    1. ]
 [ 39.   12.5 350.   10. ]
 [ 63.    4.1 850.    6. ]
 ...
 [ 58.    3.2 850.   11. ]
 [ 59.   25.6 300.   13. ]
 [ 38.   10.8 550.   45. ]]
Array shape: (990, 4)


In [ ]:
column_means = np.mean(X, axis=0)

print("Column means:", column_means.round(2))

Column means: [ 48.17   9.5  635.71  11.82]


In [ ]:
mask = appointments["lead_days"] > 14

count = mask.sum()

share = mask.mean()

print("Appointments above 14 days:", count)
print("Share:", round(share * 100, 1), "%")

Appointments above 14 days: 296
Share: 29.9 %


In [ ]:
x = appointments["distance_km"].to_numpy()

z = (x - np.mean(x)) / np.std(x)

print("First five z-scores:", z[:5].round(2))

First five z-scores: [ 1.38  0.49 -0.88 -0.32  1.54]


In [ ]:
outlier_mask = np.abs(z) > 2

outlier_count = outlier_mask.sum()


print("Distance z-score outliers:", outlier_count)

Distance z-score outliers: 53


# Task 1.5



In [ ]:
clinics = pd.read_csv("/content/clinics.csv")

print(clinics.head())

print("\nShape:", clinics.shape)

print("\nColumns:", clinics.columns.tolist())



  clinic_id             clinic_name    city    tier  latitude  longitude
0       C01   Kothrud Family Clinic    Pune   Metro   18.5074    73.8077
1       C02    Hadapsar Care Centre    Pune   Metro   18.5089    73.9260
2       C03  Sitabuldi Health Point  Nagpur  Tier-2   21.1458    79.0882
3       C04    Malviya Nagar Clinic  Jaipur  Tier-2   26.8553    75.8150
4       C05   Vaishali Nagar Clinic  Jaipur  Tier-2   26.9124    75.7873

Shape: (8, 6)

Columns: ['clinic_id', 'clinic_name', 'city', 'tier', 'latitude', 'longitude']


In [ ]:
import requests

latitude = clinics.loc[0, "latitude"]
longitude = clinics.loc[0, "longitude"]

response = requests.get(
    "https://api.open-meteo.com/v1/forecast",
    params={
        "latitude": latitude,
        "longitude": longitude,
        "hourly": "temperature_2m",
        "forecast_days": 1
    },
    timeout=10
)

print("Status code:", response.status_code)

In [ ]:
weather_data = response.json()

temperature_list = weather_data["hourly"]["temperature_2m"]

temp_c = temperature_list[0]

print("Clinic:", clinics.loc[0, "clinic_name"])
print("Temperature:", temp_c, "°C")

In [ ]:
import requests

temperatures = []

for index, clinic in clinics.iterrows():

    try:
        response = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude": clinic["latitude"],
                "longitude": clinic["longitude"],
                "hourly": "temperature_2m",
                "forecast_days": 1
            },
            timeout=10
        )

        if response.status_code != 200:
            raise ValueError(f"HTTP status {response.status_code}")

        weather_data = response.json()

        if "hourly" not in weather_data or not weather_data["hourly"].get("temperature_2m"):
            raise ValueError("Hourly temperature data missing")

        temp_c = weather_data["hourly"]["temperature_2m"][0]

    except requests.exceptions.RequestException as error:
        try:
            response = requests.get(
                "https://api.open-meteo.com/v1/forecast",
                params={
                    "latitude": clinic["latitude"],
                    "longitude": clinic["longitude"],
                    "hourly": "temperature_2m",
                    "forecast_days": 1
                },
                timeout=10
            )

            if response.status_code != 200:
                raise ValueError(f"HTTP status {response.status_code}")

            weather_data = response.json()

            if "hourly" not in weather_data or not weather_data["hourly"].get("temperature_2m"):
                raise ValueError("Hourly temperature data missing")

            temp_c = weather_data["hourly"]["temperature_2m"][0]

        except (requests.exceptions.RequestException, ValueError, KeyError) as retry_error:
            temp_c = None
            print(f"Failed for {clinic['clinic_name']}: {retry_error}")

    except (ValueError, KeyError) as error:
        temp_c = None
        print(f"Failed for {clinic['clinic_name']}: {error}")

    temperatures.append(temp_c)

    print(clinic["clinic_name"], ":", temp_c)

Kothrud Family Clinic : 15.8
Hadapsar Care Centre : 16.9
Sitabuldi Health Point : 18.4
Malviya Nagar Clinic : 15.9
Vaishali Nagar Clinic : 15.9
Edappally Medical Centre : 25.7
Vijay Nagar Clinic : 14.3
Gomti Nagar Care Point : 16.1


In [ ]:
clinics["temp_c"] = temperatures

print(clinics)

clinics.to_csv("clinics_enriched.csv", index=False)

print("\nShape:", clinics.shape)

print("\nTemperature data type:", clinics["temp_c"].dtype)
print("Missing temperatures:", clinics["temp_c"].isnull().sum())

# Task 1.6


In [ ]:
print("Unique appointment IDs:", appointments["appointment_id"].nunique())

print("Duplicate appointment IDs:", appointments["appointment_id"].duplicated().sum())

print("\nMissing values in each column:")
print(appointments.isnull().sum())

print("\nTotal missing values:", appointments.isnull().sum().sum())

print("\nTotal appointment records:", len(appointments))

Unique appointment IDs: 990
Duplicate appointment IDs: 0

Missing values in each column:
appointment_id     0
patient_id         0
clinic_id          0
booked_on          0
appt_date          0
age                0
gender             0
distance_km        0
fee_inr            0
booking_channel    0
visit_type         0
reminder_sent      0
prior_no_shows     0
status             0
lead_days          0
no_show            0
dtype: int64

Total missing values: 0

Total appointment records: 990


In [ ]:
appointments = appointments.sort_values(by="appointment_id")

appointments["booked_on"] = appointments["booked_on"].dt.strftime("%Y-%m-%d")

appointments["appt_date"] = appointments["appt_date"].dt.strftime("%Y-%m-%d")

required_columns = [
    "appointment_id", "patient_id", "clinic_id", "booked_on",
    "appt_date", "age", "gender", "distance_km", "fee_inr",
    "booking_channel", "visit_type", "reminder_sent",
    "prior_no_shows", "status", "lead_days", "no_show"
]

appointments_clean = appointments[required_columns].copy()


appointments_clean.to_csv("appointments_clean.csv", index=False)

print("Final shape:", appointments_clean.shape)
print("Columns:", appointments_clean.columns.tolist())

Final shape: (990, 16)
Columns: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status', 'lead_days', 'no_show']


In [ ]:
appointments_clean.head()

,appointment_id,patient_id,clinic_id,booked_on,appt_date,age,gender,distance_km,fee_inr,booking_channel,visit_type,reminder_sent,prior_no_shows,status,lead_days,no_show
0,A0001,P509,C05,2025-01-24,2025-01-25,60,F,18.0,850,Phone,Consultation,Yes,1,No-Show,1,1
1,A0002,P046,C07,2025-05-27,2025-06-06,39,M,12.5,350,Walk-In,Follow-up,No,2,Attended,10,0
2,A0003,P009,C02,2025-03-24,2025-03-30,63,F,4.1,850,Walk-In,Consultation,No,0,Attended,6,0
4,A0004,P482,C01,2025-04-18,2025-04-22,52,F,7.5,750,App,Consultation,Yes,0,Attended,4,0
5,A0005,P251,C01,2024-11-23,2025-01-02,64,F,19.0,700,Phone,Consultation,No,0,No-Show,40,1


In [ ]:
appointments_clean = pd.read_csv('/content/appointments_clean.csv')
clinics_enriched = pd.read_csv('/content/clinics_enriched.csv')

print("Appointments:", appointments_clean.shape)
print("Clinics:", clinics_enriched.shape)

Appointments: (990, 16)
Clinics: (8, 7)


In [ ]:
print("Clinic columns:", clinics_enriched.columns.tolist())
print("Appointment columns:", appointments_clean.columns.tolist())

Clinic columns: ['clinic_id', 'clinic_name', 'city', 'tier', 'latitude', 'longitude', 'temp_c']
Appointment columns: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status', 'lead_days', 'no_show']


In [ ]:
def format_value(value):
    """
    Convert a Python value into SQL-compatible format.

    Parameters:
        value: The input value (string, number, or missing value).

    Returns:
        str: 'NULL' for missing values,
             quoted text for strings,
             unquoted text for numbers.
    """
    if pd.isna(value):
        return "NULL"

    elif isinstance(value, str):
        return f"'{value}'"

    else:
        return str(value)

In [ ]:
with open("/content/load_data.sql", "w") as file:

    for _, row in clinics_enriched.iterrows():

        formatted_values = []

        for value in row:


            formatted_value = format_value(value)

            formatted_values.append(formatted_value)

        values = ", ".join(formatted_values)

        sql = f"INSERT INTO clinics VALUES ({values});\n"

        file.write(sql)

In [ ]:
with open("/content/load_data.sql", "a") as file:

    for _, row in appointments_clean.iterrows():

        formatted_values = []

        for value in row:

            formatted_value = format_value(value)

            formatted_values.append(formatted_value)

        values = ", ".join(formatted_values)

        sql = f"INSERT INTO appointments VALUES ({values});\n"


        file.write(sql)

In [ ]:
with open("/content/load_data.sql", "r") as file:
    sql_content = file.readlines()


print("Total INSERT statements:", len(sql_content))

Total INSERT statements: 998


In [ ]:
clinic_count = sum(
    line.startswith("INSERT INTO clinics")
    for line in sql_content
)

appointment_count = sum(
    line.startswith("INSERT INTO appointments")
    for line in sql_content
)


print("Clinic statements:", clinic_count)
print("Appointment statements:", appointment_count)
print("Total statements:", len(sql_content))

print("First statement:", sql_content[0])
print("Last statement:", sql_content[-1])

Clinic statements: 8
Appointment statements: 990
Total statements: 998
First statement: INSERT INTO clinics VALUES ('C01', 'Kothrud Family Clinic', 'Pune', 'Metro', 18.5074, 73.8077, 15.8);

Last statement: INSERT INTO appointments VALUES ('A1000', 'P085', 'C08', '2024-12-03', '2025-01-17', 38, 'M', 10.8, 550, 'App', 'Follow-up', 'Yes', 2, 'No-Show', 45, 1);



# Task 2.2

In [ ]:
import pandas as pd

appointments_clean = pd.read_csv("/content/appointments_clean.csv")

clinics_enriched = pd.read_csv("/content/clinics_enriched.csv")

In [ ]:
clinic_groups = appointments_clean.groupby("clinic_id")

appointment_counts = clinic_groups["appointment_id"].count()

print(appointment_counts)





clinic_id
C01    160
C02    143
C03     82
C04    142
C05    130
C06    142
C07    101
C08     90
Name: appointment_id, dtype: int64


In [ ]:
no_show_counts = clinic_groups["no_show"].sum()
print(no_show_counts)

clinic_id
C01    36
C02    39
C03    13
C04    38
C05    18
C06    23
C07    21
C08    30
Name: no_show, dtype: int64


In [ ]:
clinic_summary = appointments_clean.groupby("clinic_id").agg(
    appointments=("appointment_id", "count"),
    no_shows=("no_show", "sum")
)

clinic_summary["no_show_pct"] = (
    100 * clinic_summary["no_shows"] / clinic_summary["appointments"]
).round(1)

print(clinic_summary.sort_values(by="no_show_pct", ascending=False))

           appointments  no_shows  no_show_pct
clinic_id                                     
C08                  90        30         33.3
C02                 143        39         27.3
C04                 142        38         26.8
C01                 160        36         22.5
C07                 101        21         20.8
C06                 142        23         16.2
C03                  82        13         15.9
C05                 130        18         13.8
